In [1]:
import pandas as pd

# utf-8-sig: el archivo trae BOM, si no se declara queda pegado al nombre de la 1a columna
df = pd.read_csv("datos-2026-09-09.csv", encoding="utf-8-sig")

#df = df.drop(columns = ["x", "y"])
#df = df[(df["delito"] != "Abuso sexual infantil")]

# separar la fecha en dia, mes y año para ampliar los features
df[['dia', 'mes', 'año']] = df['fecha'].str.split('/', expand=True)
df = df.drop(columns = ["fecha", "zona_geografica", "clave_mun"])

df.head(20)

,delito,x,y,colonia,municipio,hora,bien_afectado,dia,mes,año
0,Abuso sexual infantil,-103.471379,20.775161,VILLA FONTANA DIAMANTE,ZAPOPAN,N.D.,La libertad y la seguridad sexual,04,02,2020
1,Abuso sexual infantil,-103.473976,20.642161,LOMAS DE LA PRIMAVERA,ZAPOPAN,N.D.,La libertad y la seguridad sexual,05,02,2020
2,Abuso sexual infantil,-103.365352,20.497948,CHULAVISTA E1,TLAJOMULCO DE ZUÑIGA,N.D.,La libertad y la seguridad sexual,05,02,2020
3,Abuso sexual infantil,-103.397761,20.601724,SANTA MARIA TEQUEPEXPAN,SAN PEDRO TLAQUEPAQUE,N.D.,La libertad y la seguridad sexual,05,02,2020
4,Abuso sexual infantil,-103.303802,20.587703,MANANTIAL,SAN PEDRO TLAQUEPAQUE,N.D.,La libertad y la seguridad sexual,05,02,2020
5,Abuso sexual infantil,-103.451766,20.625378,SANTA ANA TEPETITLAN,ZAPOPAN,N.D.,La libertad y la seguridad sexual,05,01,2020
6,Abuso sexual infantil,-103.350595,20.662749,MODERNA,GUADALAJARA,N.D.,La libertad y la seguridad sexual,05,02,2020
7,Abuso sexual infantil,-103.394773,20.602175,JARDINES DE SANTA MARIA,SAN PEDRO TLAQUEPAQUE,N.D.,La libertad y la seguridad sexual,05,02,2020
8,Abuso sexual infantil,-103.267360,20.657193,ZALATITAN CENTRO,TONALA,N.D.,La libertad y la seguridad sexual,05,02,2020
9,Abuso sexual infantil,-103.420355,20.598951,PARQUES DE SANTA MARIA,SAN PEDRO TLAQUEPAQUE,N.D.,La libertad y la seguridad sexual,06,02,2020


In [2]:
df_pob = pd.read_excel("ISDCOL2020_2021.xlsx")

df_pob.head()

,Número único de colonia,Municipios en los que se encuentra,Nombre de la colonia,Código postal,Superficie en kilómetros cuadrados,Población total,Población femenina,Población masculina,Población de 0 a 2 años,Población femenina de 0 a 2 años,...,Viviendas particulares habitadas que disponen de línea telefónica fija,Viviendas particulares habitadas que disponen de teléfono celular,Viviendas particulares habitadas que disponen de Internet,Viviendas particulares habitadas que disponen de servicio de televisión de paga,"Viviendas particulares habitadas que disponen de servicio de películas, música o videos de paga por Internet",Viviendas particulares habitadas que disponen de consola de videojuegos,Viviendas particulares habitadas sin radio ni televisor,Viviendas particulares habitadas sin línea telefónica ni teléfono celular,Viviendas particulares habitadas sin computadora ni Internet,Viviendas particulares habitadas sin tecnologías de la información y de la comunicación (TIC)
0,4461,Tototlán,REAL CAMPESTRE,47730.0,1722.57,ND2020,ND2020,ND2020,ND2020,ND2020,...,ND2020,ND2020,ND2020,ND2020,ND2020,ND2020,ND2020,ND2020,ND2020,ND2020
1,3217,Tolimán,LOMA BONITA,49750.0,2640.49,ND2020,ND2020,ND2020,ND2020,ND2020,...,ND2020,ND2020,ND2020,ND2020,ND2020,ND2020,ND2020,ND2020,ND2020,ND2020
2,4375,Zapotlán el Grande,LAS PALMAS,49078.0,2907.89,ND2020,ND2020,ND2020,ND2020,ND2020,...,ND2020,ND2020,ND2020,ND2020,ND2020,ND2020,ND2020,ND2020,ND2020,ND2020
3,3836,Sayula,EL CAMICHIN,49330.0,3101.13,ND2020,ND2020,ND2020,ND2020,ND2020,...,ND2020,ND2020,ND2020,ND2020,ND2020,ND2020,ND2020,ND2020,ND2020,ND2020
4,2541,San Pedro Tlaquepaque,ATORINO,45560.0,3105.72,ND2020,ND2020,ND2020,ND2020,ND2020,...,ND2020,ND2020,ND2020,ND2020,ND2020,ND2020,ND2020,ND2020,ND2020,ND2020


In [3]:
# todas las columnas con el mismo conteo -> no hay faltantes explicitos
df.count()

delito           306152
x                300380
y                300380
colonia          306152
municipio        306152
hora             306152
bien_afectado    306152
dia              306152
mes              306152
año              306152
dtype: int64

In [4]:
# Normalizar erratas de captura antes de clasificar
hora_txt = df["hora"].astype(str).str.strip()
# separadores mal capturados: "11;00", "15,30", "19_30" -> "11:00", "15:30", "19:30"
hora_txt = hora_txt.str.replace(r"[;,_]", ":", regex=True)
df["hora"] = hora_txt
hora_num = pd.to_numeric(hora_txt, errors="coerce")

# Cuatro formatos presentes en la columna:
es_hhmm     = hora_txt.str.contains(":", na=False)
es_nd       = hora_txt.str.contains("N.D.", case=False, na=False)   # case=False capta "n.d."
es_fraccion = hora_num.between(0, 1)          # fraccion de dia de Excel, incluye "4.17E-02"
es_hpunto   = hora_num > 1                    # "9.3" -> 9:30, "18" -> 18:00, "1515" -> 15:15

# .copy() para que cada subconjunto sea un DataFrame propio y no un slice de df
df_correct_time = df[es_hhmm].copy()
df_nd           = df[es_nd].copy()
df_dec          = df[es_fraccion].copy()
df_hpunto       = df[es_hpunto].copy()

total = df["delito"].count()
print("Porcentage CT        = %.2f%%" % (len(df_correct_time)/total*100))
print("Porcentage ND        = %.2f%%" % (len(df_nd)/total*100))
print("Porcentage Decimals  = %.2f%%" % (len(df_dec)/total*100))
print("Porcentage H.punto   = %.2f%%" % (len(df_hpunto)/total*100))
print("Sin clasificar       = %d filas" % (total - len(df_correct_time) - len(df_nd) - len(df_dec) - len(df_hpunto)))

Porcentage CT        = 87.12%
Porcentage ND        = 9.89%
Porcentage Decimals  = 2.99%
Porcentage H.punto   = 0.01%
Sin clasificar       = 0 filas


In [5]:
# grupo "HH:MM" -> hora y minuto
# errors='coerce' manda a NaT los valores invalidos (p.ej. '096:30'), que luego se descartan
df_dt = pd.to_datetime(df_correct_time["hora"], format='%H:%M', errors='coerce')
df_correct_time["hora"] = df_dt.dt.hour
df_correct_time["min"] = df_dt.dt.minute
df_correct_time = df_correct_time.dropna(subset=["hora", "min"])

df_correct_time.head()

,delito,x,y,colonia,municipio,hora,bien_afectado,dia,mes,año,min
19,Abuso sexual infantil,-103.395449,20.641250,VICTORIA,GUADALAJARA,8.0,La libertad y la seguridad sexual,10,02,2020,15.0
55,Abuso sexual infantil,-103.211973,20.665464,URBIVILLA DEL RIO,TONALA,22.0,La libertad y la seguridad sexual,17,02,2020,30.0
56,Abuso sexual infantil,-103.346717,20.679631,ZONA CENTRO,GUADALAJARA,19.0,La libertad y la seguridad sexual,17,02,2020,0.0
67,Abuso sexual infantil,-103.299334,20.646230,VICENTE GUERRERO,GUADALAJARA,17.0,La libertad y la seguridad sexual,19,02,2020,19.0
68,Abuso sexual infantil,-103.273119,20.630311,LOMAS DEL CAMICHIN,TONALA,16.0,La libertad y la seguridad sexual,19,02,2020,0.0


In [6]:
# grupo decimal: son horas de Excel como fraccion de dia
# 0.177083333 * 24 = 4.25 -> 04:15   |   4.17E-02 * 24 = 1.0 -> 01:00
# las < 0.1 (antes de las 02:24) salen exportadas en notacion cientifica
minutos_del_dia = (df_dec["hora"].astype(float) * 24 * 60).round()
df_dec["hora"] = (minutos_del_dia // 60) % 24
df_dec["min"] = minutos_del_dia % 60

df_dec.head()

,delito,x,y,colonia,municipio,hora,bien_afectado,dia,mes,año,min
130644,Robo a persona,-103.421786,20.673137,PRADOS GUADALUPE,ZAPOPAN,4.0,El patrimonio,29,11,2022,15.0
205478,Abuso sexual infantil,-103.400080,20.476089,COLINA REAL,TLAJOMULCO DE ZUÑIGA,0.0,La libertad y la seguridad sexual,25,11,2024,0.0
205479,Abuso sexual infantil,-103.244690,20.485409,LA PURISIMA,TLAJOMULCO DE ZUÑIGA,0.0,La libertad y la seguridad sexual,11,10,2024,0.0
205480,Abuso sexual infantil,-103.353636,20.589040,LOMAS DEL CUATRO,SAN PEDRO TLAQUEPAQUE,0.0,La libertad y la seguridad sexual,27,11,2024,0.0
205481,Abuso sexual infantil,-103.183707,20.554995,AGUA BLANCA,TONALA,0.0,La libertad y la seguridad sexual,09,10,2024,0.0


In [7]:
# grupo "hora.minutos" capturado con punto en lugar de dos puntos:
# "9.3" -> 09:30, "19.05" -> 19:05, "18" -> 18:00, "1515" -> 15:15
txt = df_hpunto["hora"].astype(str)

# el decimal son minutos, no fraccion: se rellena a la derecha ("3" -> "30", "05" -> "05")
df_hpunto["hora"] = txt.str.split(".").str[0].astype(int)
df_hpunto["min"] = (txt.str.split(".").str[1]
                       .fillna("0").replace("", "0")
                       .str.pad(2, side="right", fillchar="0").astype(int))

# "1515" -> 15:15 (cuatro digitos pegados, sin separador)
pegado = txt.str.fullmatch(r"\d{4}")
df_hpunto.loc[pegado, "hora"] = txt[pegado].str[:2].astype(int)
df_hpunto.loc[pegado, "min"] = txt[pegado].str[2:].astype(int)

# descartar lo que no sea una hora valida
df_hpunto = df_hpunto[(df_hpunto["hora"] <= 23) & (df_hpunto["min"] <= 59)]

df_hpunto.head()

,delito,x,y,colonia,municipio,hora,bien_afectado,dia,mes,año,min
212101,Lesiones dolosas,-103.408387,20.738406,ARCOS DE ZAPOPAN,ZAPOPAN,15,La vida y la integridad corporal,23,11,2024,15
214924,Robo a carga pesada,-103.373145,20.642543,COLON INDUSTRIAL,GUADALAJARA,10,El patrimonio,18,10,2024,0
226182,Robo a vehiculos particulares,-103.491086,20.499743,LOMAS DE SANTA ANITA,TLAJOMULCO DE ZUÑIGA,16,El patrimonio,20,11,2024,30
234837,Robo de motocicleta,-103.292595,20.530974,EL REFUGIO DEL VALLE,TLAJOMULCO DE ZUÑIGA,19,El patrimonio,09,10,2024,5
261161,Robo a negocio,-103.347724,20.681915,CENTRO BARRANQUITAS,GUADALAJARA,12,El patrimonio,23,06,2025,50


In [8]:
# hora_conocida: marca si la hora existe en el registro.
# Los "N.D." no se imputan (faltan de forma no aleatoria: 78% de los registros de
# un solo delito), se conservan con hora nula y la bandera en False.
recuperados = [df_correct_time, df_dec, df_hpunto]
for parte in recuperados:
    parte["hora_conocida"] = True
    # Int64 (nullable) en todos antes de unir: entero pero admite nulos
    parte[["hora", "min"]] = parte[["hora", "min"]].astype("Int64")

df_nd["hora_conocida"] = False
for columna in ("hora", "min"):
    df_nd[columna] = pd.Series(pd.NA, index=df_nd.index, dtype="Int64")

df_hora = pd.concat(recuperados + [df_nd])
df_hora = df_hora.sort_values(by=["hora", "min"], ascending=True)

print("Filas conservadas = %d de %d (%.2f%%)" % (len(df_hora), len(df), len(df_hora)/len(df)*100))
print(df_hora["hora_conocida"].value_counts())
df_hora.head(20)

Filas conservadas = 306142 de 306152 (100.00%)
hora_conocida
True     275878
False     30264
Name: count, dtype: int64


,delito,x,y,colonia,municipio,hora,bien_afectado,dia,mes,año,min,hora_conocida
154,Abuso sexual infantil,-103.388296,20.658690,JARDINES DEL BOSQUE,GUADALAJARA,0,La libertad y la seguridad sexual,08,03,2020,0,True
177,Abuso sexual infantil,-103.272601,20.626443,BALCONES DE SANTA CRUZ,TONALA,0,La libertad y la seguridad sexual,11,03,2020,0,True
190,Abuso sexual infantil,-103.272335,20.690488,EL BETHEL,GUADALAJARA,0,La libertad y la seguridad sexual,14,03,2020,0,True
192,Abuso sexual infantil,-103.361932,20.499305,CHULAVISTA E1,TLAJOMULCO DE ZUÑIGA,0,La libertad y la seguridad sexual,14,03,2020,0,True
198,Abuso sexual infantil,-103.348561,20.611749,MIRAVALLE,SAN PEDRO TLAQUEPAQUE,0,La libertad y la seguridad sexual,16,03,2020,0,True
384,Abuso sexual infantil,-103.311412,20.685166,SANTA MARIA,GUADALAJARA,0,La libertad y la seguridad sexual,14,01,2020,0,True
769,Abuso sexual infantil,-103.375333,20.797915,SAN ESTEBAN,ZAPOPAN,0,La libertad y la seguridad sexual,02,09,2020,0,True
881,Abuso sexual infantil,-103.281984,20.605183,SAN PEDRITO,SAN PEDRO TLAQUEPAQUE,0,La libertad y la seguridad sexual,25,01,2020,0,True
1013,Abuso sexual infantil,-103.363248,20.663229,MODERNA,GUADALAJARA,0,La libertad y la seguridad sexual,05,11,2020,0,True
1088,Abuso sexual infantil,-103.379816,20.632632,LOMAS DE POLANCO,GUADALAJARA,0,La libertad y la seguridad sexual,04,01,2020,0,True


In [9]:
# Poblacion de la colonia (censo 2020 por colonia, ISDCOL2020_2021.csv)
import numpy as np
import unicodedata

def normalizar(serie):
    """Llave de union: mayusculas, sin acentos y sin espacios repetidos.
    ISDCOL trae 'Tlajomulco de Zuniga' y los delitos 'TLAJOMULCO DE ZUNIGA'."""
    s = serie.astype(str).str.strip().str.upper()
    s = s.map(lambda x: "".join(ch for ch in unicodedata.normalize("NFD", x)
                                if unicodedata.category(ch) != "Mn"))
    return s.str.replace(r"\s+", " ", regex=True)

# columna del censo -> nombre de trabajo. Los conteos crudos que no van al dataset final
# solo sirven de numerador/denominador de las proporciones de abajo
columnas_censo = {
    "Poblacion total":               "poblacion_colonia",
    "Poblacion femenina":            "poblacion_femenina",
    "Poblacion masculina":           "poblacion_masculina",
    "Total de viviendas habitadas":  "Total de viviendas habitadas",
    "Grado promedio de escolaridad": "Grado promedio de escolaridad",
    "Viviendas particulares habitadas que disponen de Internet":
        "Viviendas particulares habitadas que disponen de Internet",
    "Poblacion de 12 años y más desocupada":            "desocupada",
    "Poblacion de 12 años y más economicamente activa": "pea",
    "Poblacion sin afiliacion a servicios de salud":    "sin_salud",
    "Viviendas particulares habitadas que no disponen de automovil o camioneta, "
    "ni de motocicleta o motoneta":                     "sin_vehiculo",
    # denominador de Internet/vehiculo: con "Viviendas particulares habitadas" (sin "Total")
    # 302 filas del censo dan proporciones > 1; con el total ninguna
    "Total de viviendas particulares habitadas":        "viv_part_habitadas",
    "Poblacion de 18 a 24 años":                        "pob_18_24",
    "Hogares censales con persona de referencia mujer": "hogares_jefa_mujer",
    "Total de hogares censales":                        "hogares",
    "Viviendas particulares deshabitadas":              "viv_deshabitadas",
    "Total de viviendas":                               "viviendas_total",
    "Poblacion nacida en otra entidad":                 "nacidos_otra_entidad",
    # el encabezado dice km2 pero los valores son m2 (209,238 para una colonia de 2,877 hab.)
    "Superficie en kilometros cuadrados":               "superficie_m2",
    "Promedio de ocupantes por cuarto en viviendas particulares habitadas":
        "ocupantes_por_cuarto",
}

# son promedios, no totales: sumarlos al juntar CPs de una misma colonia no tiene sentido
columnas_promedio = {"Grado promedio de escolaridad", "ocupantes_por_cuarto"}

# algunos encabezados traen saltos de linea ("Viviendas particulares\nhabitadas que no\n...")
def limpiar_encabezado(nombre):
    return " ".join(nombre.split())

# cp850: este csv NO es utf-8, la enye viene en el byte 0xA4 (codepage de DOS)
censo = pd.read_csv("ISDCOL2020_2021.csv", encoding="cp850", low_memory=False,
                    usecols=lambda c: limpiar_encabezado(c) in
                        {"Municipios en los que se encuentra", "Colonia", *columnas_censo})
censo.columns = [limpiar_encabezado(c) for c in censo.columns]
censo = censo.rename(columns={**columnas_censo,
                              "Municipios en los que se encuentra": "municipio",
                              "Colonia": "colonia"})

# "2,877" -> 2877 ; los "ND2020" quedan como nulos
for destino in columnas_censo.values():
    censo[destino] = pd.to_numeric(censo[destino].astype(str).str.replace(",", ""),
                                   errors="coerce")

# una colonia puede cruzar municipios: "GUADALAJARA,SAN PEDRO TLAQUEPAQUE" -> una fila por municipio
censo["col_k"] = normalizar(censo["colonia"])
censo["mun_k"] = normalizar(censo["municipio"]).str.split(",")
censo = censo.explode("mun_k")
censo["mun_k"] = censo["mun_k"].str.strip()

# hay nombres de colonia repetidos dentro del mismo municipio (distinto CP); como el
# registro del delito solo trae el nombre, se agregan: el nombre refiere al area completa.
# los totales se suman; los promedios se promedian
agregacion = {destino: (lambda s: s.mean()) if destino in columnas_promedio
                       else (lambda s: s.sum(min_count=1))
              for destino in columnas_censo.values()}
cpc = censo.groupby(["mun_k", "col_k"])[list(columnas_censo.values())].agg(agregacion)

# proporciones: los conteos crudos escalan con el tamano de la colonia (corr ~0.99 con
# poblacion); divididos describen como es la colonia y no cuanta gente tiene.
# Se calculan despues de sumar los CPs: cociente de sumas, no promedio de cocientes
proporciones = pd.DataFrame({
    "tasa_desempleo":              cpc["desocupada"] / cpc["pea"],
    "prop_sin_salud":              cpc["sin_salud"] / cpc["poblacion_colonia"],
    "prop_sin_vehiculo":           cpc["sin_vehiculo"] / cpc["viv_part_habitadas"],
    "prop_internet":               cpc["Viviendas particulares habitadas que disponen de Internet"]
                                   / cpc["viv_part_habitadas"],
    "prop_18_24":                  cpc["pob_18_24"] / cpc["poblacion_colonia"],
    # definicion INEGI: hombres por cada 100 mujeres
    "relacion_hombres_mujeres":    100 * cpc["poblacion_masculina"] / cpc["poblacion_femenina"],
    "prop_jefatura_femenina":      cpc["hogares_jefa_mujer"] / cpc["hogares"],
    "prop_viviendas_deshabitadas": cpc["viv_deshabitadas"] / cpc["viviendas_total"],
    "prop_nacidos_otra_entidad":   cpc["nacidos_otra_entidad"] / cpc["poblacion_colonia"],
    "densidad_hab_km2":            cpc["poblacion_colonia"] / (cpc["superficie_m2"] / 1e6),
    "ocupantes_por_cuarto":        cpc["ocupantes_por_cuarto"],
}).replace([np.inf, -np.inf], np.nan)   # denominador 0 -> sin dato

columnas_enteras = ["poblacion_colonia", "poblacion_femenina", "poblacion_masculina",
                    "Total de viviendas habitadas",
                    "Viviendas particulares habitadas que disponen de Internet"]
tabla_censo = pd.concat([cpc[columnas_enteras + ["Grado promedio de escolaridad"]],
                         proporciones], axis=1)

claves = pd.MultiIndex.from_arrays([normalizar(df_hora["municipio"]),
                                    normalizar(df_hora["colonia"])])
for columna in tabla_censo.columns:
    df_hora[columna] = claves.map(tabla_censo[columna])
    if columna in columnas_enteras:
        df_hora[columna] = df_hora[columna].astype("Int64")

for columna in tabla_censo.columns:
    con_dato = df_hora[columna].notna()
    print("%-60s %6d de %d (%.2f%%)" % (columna, con_dato.sum(), len(df_hora), con_dato.mean()*100))
print()
print(df_hora[list(proporciones.columns)].describe().T.to_string())
df_hora[["colonia", "municipio", "poblacion_colonia"] + list(proporciones.columns)].head(10)

poblacion_colonia                                            282048 de 306142 (92.13%)
poblacion_femenina                                           282048 de 306142 (92.13%)
poblacion_masculina                                          282048 de 306142 (92.13%)
Total de viviendas habitadas                                 282048 de 306142 (92.13%)
Viviendas particulares habitadas que disponen de Internet    282048 de 306142 (92.13%)
Grado promedio de escolaridad                                280739 de 306142 (91.70%)
tasa_desempleo                                               282048 de 306142 (92.13%)
prop_sin_salud                                               282048 de 306142 (92.13%)
prop_sin_vehiculo                                            282048 de 306142 (92.13%)
prop_internet                                                282048 de 306142 (92.13%)
prop_18_24                                                   282048 de 306142 (92.13%)
relacion_hombres_mujeres                   

                                count         mean          std       min          25%          50%           75%           max
tasa_desempleo               282048.0     0.018854     0.067594   0.00000     0.010404     0.013740      0.017057      1.000000
prop_sin_salud               282048.0     0.293971     0.084134   0.00000     0.231567     0.289877      0.348910      0.833333
prop_sin_vehiculo            282048.0     0.398125     0.160408   0.00000     0.290076     0.440789      0.507848      1.000000
prop_internet                282048.0     0.703486     0.156772   0.00000     0.613915     0.705078      0.824527      1.000000
prop_18_24                   282048.0     0.120876     0.021769   0.00000     0.111421     0.119729      0.130912      0.602513
relacion_hombres_mujeres     282040.0    96.445508    39.712943  40.00000    92.031933    95.704129     98.022847   2296.875000
prop_jefatura_femenina       282048.0     0.364417     0.074889   0.00000     0.323830     0.362287     

,colonia,municipio,poblacion_colonia,tasa_desempleo,prop_sin_salud,prop_sin_vehiculo,prop_internet,prop_18_24,relacion_hombres_mujeres,prop_jefatura_femenina,prop_viviendas_deshabitadas,prop_nacidos_otra_entidad,densidad_hab_km2,ocupantes_por_cuarto
154,JARDINES DEL BOSQUE,GUADALAJARA,6068,0.013972,0.259888,0.179076,0.885485,0.109756,88.760537,0.400566,0.111502,0.188036,4062.987937,0.570
177,BALCONES DE SANTA CRUZ,TONALA,225,0.008197,0.342222,0.339623,0.547170,0.168889,90.677966,0.264151,0.129032,0.084444,17569.450084,1.180
190,EL BETHEL,GUADALAJARA,8987,0.009897,0.400134,0.551485,0.702970,0.129854,97.040123,0.305446,0.066488,0.253143,20985.109749,1.150
192,CHULAVISTA E1,TLAJOMULCO DE ZUÑIGA,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
198,MIRAVALLE,SAN PEDRO TLAQUEPAQUE,24199,0.017954,0.190421,0.525601,0.708165,0.115666,90.917699,0.401552,0.049759,0.103186,17041.008184,0.925
384,SANTA MARIA,GUADALAJARA,10550,0.011258,0.341137,0.477382,0.710075,0.116398,94.901164,0.411241,0.095238,0.076398,14247.656740,0.840
769,SAN ESTEBAN,ZAPOPAN,1736,0.012658,0.217166,0.289037,0.192691,0.055300,100.461894,0.342193,0.071429,0.013249,1119.753582,1.520
881,SAN PEDRITO,SAN PEDRO TLAQUEPAQUE,14689,0.012111,0.420383,0.542746,0.588415,0.130097,97.749023,0.352548,0.134525,0.084008,8509.833049,1.100
1013,MODERNA,GUADALAJARA,4015,0.016722,0.199502,0.374911,0.797577,0.100125,92.979872,0.396294,0.178552,0.157659,2246.031323,0.630
1088,LOMAS DE POLANCO,GUADALAJARA,21615,0.019825,0.296229,0.496565,0.699489,0.116539,93.560233,0.383653,0.076468,0.105344,16259.220921,0.910


In [10]:
# Dataframe final: dia/mes/año venian como texto ('08', '03', '2020') del str.split
for columna in ("dia", "mes", "año"):
    df_hora[columna] = df_hora[columna].astype("Int64")

# columnas agrupadas por tema: que, donde, cuando, contexto
df_hora = df_hora[["delito", "bien_afectado",
                   "colonia", "municipio", "x", "y",
                   "año", "mes", "dia", "hora", "min", "hora_conocida",
                   "poblacion_colonia", "poblacion_femenina", "poblacion_masculina",
                   "Total de viviendas habitadas", "Grado promedio de escolaridad",
                   "Viviendas particulares habitadas que disponen de Internet",
                   *proporciones.columns]]

print(df_hora.dtypes.to_string())
print("")
print("%d filas x %d columnas" % df_hora.shape)
df_hora.head(10)

delito                                                        object
bien_afectado                                                 object
colonia                                                       object
municipio                                                     object
x                                                            float64
y                                                            float64
año                                                            Int64
mes                                                            Int64
dia                                                            Int64
hora                                                           Int64
min                                                            Int64
hora_conocida                                                   bool
poblacion_colonia                                              Int64
poblacion_femenina                                             Int64
poblacion_masculina               

,delito,bien_afectado,colonia,municipio,x,y,año,mes,dia,hora,...,prop_sin_salud,prop_sin_vehiculo,prop_internet,prop_18_24,relacion_hombres_mujeres,prop_jefatura_femenina,prop_viviendas_deshabitadas,prop_nacidos_otra_entidad,densidad_hab_km2,ocupantes_por_cuarto
154,Abuso sexual infantil,La libertad y la seguridad sexual,JARDINES DEL BOSQUE,GUADALAJARA,-103.388296,20.658690,2020,3,8,0,...,0.259888,0.179076,0.885485,0.109756,88.760537,0.400566,0.111502,0.188036,4062.987937,0.570
177,Abuso sexual infantil,La libertad y la seguridad sexual,BALCONES DE SANTA CRUZ,TONALA,-103.272601,20.626443,2020,3,11,0,...,0.342222,0.339623,0.547170,0.168889,90.677966,0.264151,0.129032,0.084444,17569.450084,1.180
190,Abuso sexual infantil,La libertad y la seguridad sexual,EL BETHEL,GUADALAJARA,-103.272335,20.690488,2020,3,14,0,...,0.400134,0.551485,0.702970,0.129854,97.040123,0.305446,0.066488,0.253143,20985.109749,1.150
192,Abuso sexual infantil,La libertad y la seguridad sexual,CHULAVISTA E1,TLAJOMULCO DE ZUÑIGA,-103.361932,20.499305,2020,3,14,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
198,Abuso sexual infantil,La libertad y la seguridad sexual,MIRAVALLE,SAN PEDRO TLAQUEPAQUE,-103.348561,20.611749,2020,3,16,0,...,0.190421,0.525601,0.708165,0.115666,90.917699,0.401552,0.049759,0.103186,17041.008184,0.925
384,Abuso sexual infantil,La libertad y la seguridad sexual,SANTA MARIA,GUADALAJARA,-103.311412,20.685166,2020,1,14,0,...,0.341137,0.477382,0.710075,0.116398,94.901164,0.411241,0.095238,0.076398,14247.656740,0.840
769,Abuso sexual infantil,La libertad y la seguridad sexual,SAN ESTEBAN,ZAPOPAN,-103.375333,20.797915,2020,9,2,0,...,0.217166,0.289037,0.192691,0.055300,100.461894,0.342193,0.071429,0.013249,1119.753582,1.520
881,Abuso sexual infantil,La libertad y la seguridad sexual,SAN PEDRITO,SAN PEDRO TLAQUEPAQUE,-103.281984,20.605183,2020,1,25,0,...,0.420383,0.542746,0.588415,0.130097,97.749023,0.352548,0.134525,0.084008,8509.833049,1.100
1013,Abuso sexual infantil,La libertad y la seguridad sexual,MODERNA,GUADALAJARA,-103.363248,20.663229,2020,11,5,0,...,0.199502,0.374911,0.797577,0.100125,92.979872,0.396294,0.178552,0.157659,2246.031323,0.630
1088,Abuso sexual infantil,La libertad y la seguridad sexual,LOMAS DE POLANCO,GUADALAJARA,-103.379816,20.632632,2020,1,4,0,...,0.296229,0.496565,0.699489,0.116539,93.560233,0.383653,0.076468,0.105344,16259.220921,0.910


In [11]:
df_hora.to_csv('Crime_Population.csv', index=False)